# Day 09- Populating ChromaDB Database

Today we will learn how to properly handle metadata and load our embeddings into a vector DB for retrieval from our chatbot in `chatbot.py`.

In [6]:
from pathlib import Path
import json
import chromadb

root=Path(r"C:\Users\micro\Documents\ABTalksAI-Cohort")
jsonroot = root / "knowledge_base_embed.jsonl"
db_path = root / "chroma_db"

client = chromadb.PersistentClient(path=db_path)
collection = client.get_collection(name="coverage_kb")

ids_batch = []
embeddings_batch = []
documents_batch = []
metadatas_batch = []

with open(jsonroot, "r", encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)

        ids_batch.append(str(record["id"]))
        embeddings_batch.append(record["embeddings"])
        documents_batch.append(record["text"]) 
        meta_dict = record.copy()
        for key in ["id", "embedding", "content"]:
            meta_dict.pop(key, None)
        metadatas_batch.append(meta_dict)

    collection.upsert(
        ids=ids_batch,
        embeddings=embeddings_batch,
        documents=documents_batch,
        metadatas=metadatas_batch
        )  

In [7]:
collection.count()

42

In [8]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

raw_query = "Is physical therapy covered under the Silver plan?"
emb_query = model.encode(raw_query)

results = collection.query(
    query_embeddings=[emb_query],
    n_results=5,
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11792.37it/s]


In [9]:
print(results["documents"])

[['Silver HMO (P102)\nLinked claims: 2. Status counts - Denied: 1, Approved: 1. Procedures - X-ray, Surgery.\nBronze HMO (P103)\nLinked claims: 1. Status counts - Pending: 1. Procedures - X-ray.\n8. Recommended production schema extensions\n9. Data dictionary\nplans.plan_id: Unique plan identifier and join key.\nplans.plan_name: Human-readable plan name.\nplans.monthly_premium: Monthly premium amount.\nplans.annual_deductible: Annual deductible amount.', '- Do not infer coverage from procedure name or status alone.\n6. Supplied synthetic claim register\nThe plan cost-share percentage is shown only as reference data. The file does not provide enough information to calculate member responsibility, because deductible accumulation, allowed amount, service rules, and payment details are absent.\n7. Plan-specific claim observations\nGold PPO (P101)\nLinked claims: 2. Status counts - Pending: 1, Approved: 1. Procedures - X-ray, Surgery.\nSilver HMO (P102)', 'Plan: Silver HMO. Premium: $300/mo

In [11]:
plan_results = collection.query(
    query_embeddings=[emb_query],
    n_results=5,
    where={"plan_type": "Silver HMO"},
)

print(plan_results["documents"])

[['Silver HMO (P102)\nLinked claims: 2. Status counts - Denied: 1, Approved: 1. Procedures - X-ray, Surgery.\nBronze HMO (P103)\nLinked claims: 1. Status counts - Pending: 1. Procedures - X-ray.\n8. Recommended production schema extensions\n9. Data dictionary\nplans.plan_id: Unique plan identifier and join key.\nplans.plan_name: Human-readable plan name.\nplans.monthly_premium: Monthly premium amount.\nplans.annual_deductible: Annual deductible amount.', '- Do not infer coverage from procedure name or status alone.\n6. Supplied synthetic claim register\nThe plan cost-share percentage is shown only as reference data. The file does not provide enough information to calculate member responsibility, because deductible accumulation, allowed amount, service rules, and payment details are absent.\n7. Plan-specific claim observations\nGold PPO (P101)\nLinked claims: 2. Status counts - Pending: 1, Approved: 1. Procedures - X-ray, Surgery.\nSilver HMO (P102)', 'Plan: Silver HMO. Premium: $300/mo